In [ ]:
import gc
import joblib

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.spatial import cKDTree

import matplotlib.pyplot as plt
import seaborn as sns

from category_encoders import TargetEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

import lightgbm as lgb
import xgboost as xgb

In [3]:
df = pd.read_csv("/kaggle/input/competitions/structural-damage-classification/train.csv")
df = df[df['age_building'] != 999]

num_cols = df.select_dtypes(include=['number']).columns

cols_to_check = ['building_id', 'district_id', 'vdcmun_id', 'ward_id']

for col in cols_to_check:
    df[col] = df[col].astype(str)

# Gunakan 'object', 'string', dan 'category' (bukan 'str')
cat_cols = df.select_dtypes(include=['object', 'string', 'category']).columns
cat_cols = cat_cols.drop('building_id', errors='ignore')

df = df.drop(['building_id'], axis=1)
df


,district_id,vdcmun_id,ward_id,count_floors_pre_eq,age_building,plinth_area_sq_ft,height_ft_pre_eq,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,damage_grade,superstructure
0,28,2802,280202,3,42,504,21,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Attached-2 side,Rectangular,Grade 5,has_superstructure_mud_mortar_stone
1,31,3104,310416,2,8,380,16,Flat,Cement-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,Timber-Planck,Not attached,Rectangular,Grade 1,has_superstructure_cement_mortar_brick
2,31,3111,311106,2,37,375,16,Moderate slope,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,Grade 5,has_superstructure_mud_mortar_stone
3,28,2802,280206,2,70,319,15,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Attached-1 side,Rectangular,Grade 3,has_superstructure_adobe_mud
4,29,2904,290404,1,25,495,6,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Brick/Stone,Not applicable,Not attached,Rectangular,Grade 4,has_superstructure_mud_mortar_stone
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533460,22,2206,220604,3,22,320,25,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Heavy roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,Grade 5,has_superstructure_mud_mortar_stone
533461,36,3611,361103,3,22,500,24,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,Grade 4,has_superstructure_mud_mortar_stone
533462,30,3003,300306,1,8,100,9,Moderate slope,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,Not applicable,Attached-1 side,Rectangular,Grade 3,has_superstructure_mud_mortar_stone
533463,28,2803,280303,2,2,877,23,Flat,Mud mortar-Stone/Brick,Bamboo/Timber-Light roof,Mud,TImber/Bamboo-Mud,Not attached,Rectangular,Grade 3,has_superstructure_mud_mortar_stone


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('damage_grade', axis=1),
    df['damage_grade'],
    test_size=0.2,
    stratify=df['damage_grade'],
    random_state=42)

In [ ]:
df_train_raw = pd.read_csv("/kaggle/input/competitions/structural-damage-classification/train.csv")
df_test_raw = pd.read_csv("/kaggle/input/competitions/structural-damage-classification/test.csv")

def ensure_building_id(X_tr, X_te, df_raw):
    X_tr_out = X_tr.copy()
    X_te_out = X_te.copy()
    if 'building_id' not in X_tr_out.columns:
        # Cek apakah index asli dari train.csv masih tersimpan
        if (df_raw.loc[X_tr_out.index, 'age_building'].values == X_tr_out['age_building'].values).all():
            X_tr_out['building_id'] = df_raw.loc[X_tr_out.index, 'building_id'].values
            X_te_out['building_id'] = df_raw.loc[X_te_out.index, 'building_id'].values
        else:
            # Jika sempat di-reset_index setelah filter age_building != 999
            df_clean = df_raw[df_raw['age_building'] != 999].reset_index(drop=True)
            X_tr_out['building_id'] = df_clean.loc[X_tr_out.index, 'building_id'].values
            X_te_out['building_id'] = df_clean.loc[X_te_out.index, 'building_id'].values
    return X_tr_out, X_te_out

X_train_with_id, X_test_with_id = ensure_building_id(X_train, X_test, df_train_raw)
y_train_num = y_train.astype(str).str.replace('Grade ', '').astype(int) - 1
y_test_num = y_test.astype(str).str.replace('Grade ', '').astype(int) - 1
y_test_formatted = ['Grade ' + str(p + 1) for p in y_test_num]

print("Contoh building_id berhasil dipulihkan:", X_train_with_id['building_id'].iloc[:3].tolist())

In [ ]:
class MicroStreetPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.cat_dtypes_ = {}
        self.feature_names_ = []
        self.street_lookup_ = None
        self.ward_stats_ = {}
        self.old_ward_stats_ = {}
        self.vdcmun_counts_ = {}
        self.old_vdc_counts_ = {}
        self.global_means_ = {}

    def _decode_building_id(self, df):
        b_id = df['building_id'].astype(np.int64)
        df['old_vdc_id'] = (b_id // 100_000_000).astype(str)
        df['old_ward_id'] = (b_id // 1_000_000).astype(str)
        df['old_ward_num'] = (b_id // 1_000_000).astype(np.int64)
        df['street_block_id'] = (b_id // 1_000).astype(str)
        df['survey_seq'] = (b_id % 1_000_000).astype(np.float64)
        return df

    def _clean_and_engineer(self, X):
        X_clean = X.copy()
        X_clean = self._decode_building_id(X_clean)
        
        # Fitur Fisik & Geometri Teknik Sipil
        X_clean['height_area_ratio'] = X_clean['height_ft_pre_eq'] / (X_clean['plinth_area_sq_ft'] + 1)
        X_clean['slenderness_ratio'] = X_clean['height_ft_pre_eq'] / np.sqrt(X_clean['plinth_area_sq_ft'] + 1)
        X_clean['building_volume'] = X_clean['height_ft_pre_eq'] * X_clean['plinth_area_sq_ft']
        X_clean['avg_floor_height'] = X_clean['height_ft_pre_eq'] / (X_clean['count_floors_pre_eq'] + 1)
        X_clean['area_per_floor'] = X_clean['plinth_area_sq_ft'] / (X_clean['count_floors_pre_eq'] + 1)
        X_clean['total_floor_area'] = X_clean['plinth_area_sq_ft'] * X_clean['count_floors_pre_eq']
        X_clean['is_very_old'] = (X_clean['age_building'] > 50).astype(int)
        X_clean['age_per_floor'] = X_clean['age_building'] / (X_clean['count_floors_pre_eq'] + 1)
        
        if 'plan_configuration' in X_clean.columns:
            X_clean['plan_configuration'] = X_clean['plan_configuration'].apply(
                lambda x: x if x in ['Rectangular', 'Square', 'L-shape'] else 'Others'
            )
        if 'ground_floor_type' in X_clean.columns:
            X_clean['ground_floor_type'] = X_clean['ground_floor_type'].replace({'Timber': 'Other'})
            
        # Interaksi Material Struktur
        X_clean['foundation_superstructure'] = (
            X_clean['foundation_type'].astype(str) + "_" + X_clean['superstructure'].astype(str)
        )
        X_clean['roof_floor_combo'] = (
            X_clean['roof_type'].astype(str) + "_" + 
            X_clean['ground_floor_type'].astype(str) + "_" + 
            X_clean['other_floor_type'].astype(str)
        )
        X_clean['surface_position_combo'] = (
            X_clean['land_surface_condition'].astype(str) + "_" + X_clean['position'].astype(str)
        )
        return X_clean

    def _build_census_street_lookup(self, X_census):
        # Urutkan seluruh 760.000 bangunan berdasarkan building_id (urutan jalan survei!)
        df_s = X_census[['building_id', 'old_ward_id', 'survey_seq', 'height_ft_pre_eq', 
                         'age_building', 'plinth_area_sq_ft', 'count_floors_pre_eq', 
                         'superstructure', 'foundation_type']].drop_duplicates('building_id').sort_values('building_id').reset_index(drop=True)
        
        # Ganti age 999 dengan NaN sementara agar tidak merusak rata-rata jalan
        df_s['age_clean'] = df_s['age_building'].where(df_s['age_building'] < 999, np.nan)
        
        grp = df_s.groupby('old_ward_id', sort=False)
        
        # Jarak nomor urut dengan rumah tetangga kiri dan kanan di jalan yang sama
        df_s['seq_diff_prev'] = grp['survey_seq'].diff(1).fillna(999.0).clip(0, 1000)
        df_s['seq_diff_next'] = (-grp['survey_seq'].diff(-1)).fillna(999.0).clip(0, 1000)
        
        # Posisi relatif rumah di dalam rute survei kelurahan (0.0 s.d. 1.0)
        w_min = grp['survey_seq'].transform('min')
        w_max = grp['survey_seq'].transform('max')
        df_s['survey_seq_rel'] = (df_s['survey_seq'] - w_min) / (w_max - w_min + 1.0)
        
        # Rata-rata fisik 5 rumah berurutan di jalan yang sama (window=5, center=True)
        df_s['street5_mean_height'] = grp['height_ft_pre_eq'].transform(lambda s: s.rolling(5, min_periods=1, center=True).mean())
        df_s['street5_mean_area'] = grp['plinth_area_sq_ft'].transform(lambda s: s.rolling(5, min_periods=1, center=True).mean())
        df_s['street5_mean_age'] = grp['age_clean'].transform(lambda s: s.rolling(5, min_periods=1, center=True).mean()).fillna(df_s['age_building'])
        df_s['street5_mean_floors'] = grp['count_floors_pre_eq'].transform(lambda s: s.rolling(5, min_periods=1, center=True).mean())
        
        # Apakah material rumah sama dengan tetangga tepat di sebelah kiri/kanannya?
        prev_super = grp['superstructure'].shift(1)
        next_super = grp['superstructure'].shift(-1)
        df_s['same_super_neighbors'] = ((df_s['superstructure'] == prev_super).astype(int) + 
                                        (df_s['superstructure'] == next_super).astype(int))
        
        prev_found = grp['foundation_type'].shift(1)
        next_found = grp['foundation_type'].shift(-1)
        df_s['same_found_neighbors'] = ((df_s['foundation_type'] == prev_found).astype(int) + 
                                        (df_s['foundation_type'] == next_found).astype(int))
        
        cols_keep = ['building_id', 'seq_diff_prev', 'seq_diff_next', 'survey_seq_rel',
                     'street5_mean_height', 'street5_mean_area', 'street5_mean_age', 
                     'street5_mean_floors', 'same_super_neighbors', 'same_found_neighbors']
        return df_s[cols_keep].set_index('building_id')

    def _add_spatial_features(self, X_clean):
        ward_key = X_clean['ward_id'].astype(str)
        old_ward_key = X_clean['old_ward_id'].astype(str)
        vdcmun_key = X_clean['vdcmun_id'].astype(str)
        old_vdc_key = X_clean['old_vdc_id'].astype(str)
        
        # Kepadatan makro & mikro
        X_clean['ward_building_count'] = ward_key.map(self.ward_stats_['count']).fillna(1).astype(float)
        X_clean['old_ward_building_count'] = old_ward_key.map(self.old_ward_stats_['count']).fillna(1).astype(float)
        X_clean['vdcmun_building_count'] = vdcmun_key.map(self.vdcmun_counts_).fillna(1).astype(float)
        X_clean['old_vdc_building_count'] = old_vdc_key.map(self.old_vdc_counts_).fillna(1).astype(float)
        
        # Statistik fisik per old_ward_id (resolusi mikro) & ward_id
        ow_h = old_ward_key.map(self.old_ward_stats_['height']).fillna(self.global_means_['height']).astype(float)
        ow_a = old_ward_key.map(self.old_ward_stats_['age']).fillna(self.global_means_['age']).astype(float)
        ow_p = old_ward_key.map(self.old_ward_stats_['area']).fillna(self.global_means_['area']).astype(float)
        
        w_h = ward_key.map(self.ward_stats_['height']).fillna(self.global_means_['height']).astype(float)
        w_a = ward_key.map(self.ward_stats_['age']).fillna(self.global_means_['age']).astype(float)
        
        X_clean['old_ward_mean_height'] = ow_h
        X_clean['old_ward_mean_age'] = ow_a
        X_clean['old_ward_mean_area'] = ow_p
        X_clean['height_vs_old_ward'] = X_clean['height_ft_pre_eq'] / (ow_h + 1)
        X_clean['age_vs_old_ward'] = X_clean['age_building'] / (ow_a + 1)
        X_clean['area_vs_old_ward'] = X_clean['plinth_area_sq_ft'] / (ow_p + 1)
        X_clean['height_vs_ward_mean'] = X_clean['height_ft_pre_eq'] / (w_h + 1)
        X_clean['age_vs_ward_mean'] = X_clean['age_building'] / (w_a + 1)
        
        # Gabungkan fitur tetangga jalan dari lookup building_id
        b_ids = X_clean['building_id'].astype(np.int64)
        street_feats = self.street_lookup_.reindex(b_ids)
        for col in self.street_lookup_.columns:
            X_clean[col] = street_feats[col].values
            
        # Rasio bangunan terhadap 5 rumah tetangga terdekat di jalan yang sama
        X_clean['height_vs_street5'] = X_clean['height_ft_pre_eq'] / (X_clean['street5_mean_height'] + 1)
        X_clean['area_vs_street5'] = X_clean['plinth_area_sq_ft'] / (X_clean['street5_mean_area'] + 1)
        X_clean['age_vs_street5'] = X_clean['age_building'] / (X_clean['street5_mean_age'] + 1)
        
        # Buang building_id mentah setelah semua komponennya diekstrak
        X_clean = X_clean.drop(['building_id'], axis=1, errors='ignore')
        return X_clean

    def fit(self, X, y=None, X_extra=None):
        X_temp = self._clean_and_engineer(X)
        if X_extra is not None:
            X_extra_clean = self._clean_and_engineer(X_extra)
            X_census = pd.concat([X_temp, X_extra_clean], axis=0, ignore_index=True)
        else:
            X_census = X_temp
            
        X_valid_age = X_census[X_census['age_building'] < 999]
        
        self.global_means_ = {
            'height': float(X_census['height_ft_pre_eq'].mean()),
            'age': float(X_valid_age['age_building'].mean()),
            'area': float(X_census['plinth_area_sq_ft'].mean())
        }
        
        w_key = X_census['ward_id'].astype(str)
        ow_key = X_census['old_ward_id'].astype(str)
        
        self.ward_stats_ = {
            'count': w_key.value_counts().to_dict(),
            'height': X_census.groupby(w_key)['height_ft_pre_eq'].mean().to_dict(),
            'age': X_valid_age.groupby(X_valid_age['ward_id'].astype(str))['age_building'].mean().to_dict()
        }
        self.old_ward_stats_ = {
            'count': ow_key.value_counts().to_dict(),
            'height': X_census.groupby(ow_key)['height_ft_pre_eq'].mean().to_dict(),
            'age': X_valid_age.groupby(X_valid_age['old_ward_id'].astype(str))['age_building'].mean().to_dict(),
            'area': X_census.groupby(ow_key)['plinth_area_sq_ft'].mean().to_dict()
        }
        self.vdcmun_counts_ = X_census['vdcmun_id'].astype(str).value_counts().to_dict()
        self.old_vdc_counts_ = X_census['old_vdc_id'].astype(str).value_counts().to_dict()
        
        # Bangun tabel tetangga jalan untuk seluruh sensus
        self.street_lookup_ = self._build_census_street_lookup(X_census)
        
        X_temp = self._add_spatial_features(X_temp)
        
        cat_cols = X_temp.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
        self.cat_dtypes_ = {}
        for col in cat_cols:
            unique_cats = X_temp[col].astype(str).unique()
            self.cat_dtypes_[col] = pd.CategoricalDtype(categories=unique_cats, ordered=False)
            
        self.feature_names_ = X_temp.columns.tolist()
        return self

    def transform(self, X):
        X_clean = self._clean_and_engineer(X)
        X_clean = self._add_spatial_features(X_clean)
        for col, cat_dtype in self.cat_dtypes_.items():
            if col in X_clean.columns:
                X_clean[col] = X_clean[col].astype(str).astype(cat_dtype)
        return X_clean[self.feature_names_]

# 3. MODEL SPESIALIS KE-3: STREET-NEIGHBOR SPATIAL KNN (cKDTree)
class StreetNeighborKNN:
    def __init__(self, k_street=20, k_struct=15):
        self.k_street = k_street
        self.k_struct = k_struct
        
    def fit(self, X_raw, y_num):
        b_id = X_raw['building_id'].astype(np.int64).values
        old_ward = b_id // 1_000_000
        seq = (b_id % 1_000_000).astype(np.float64)
        super_str = X_raw['superstructure'].astype(str).values
        
        # Peta ID unik ke integer kecil agar presisi float64 di cKDTree 100% akurat
        unique_wards = np.unique(old_ward)
        self.ward_map_ = {w: i for i, w in enumerate(unique_wards)}
        w_idx = np.array([self.ward_map_[w] for w in old_ward], dtype=np.float64)
        
        struct_keys = [f"{w}_{s}" for w, s in zip(old_ward, super_str)]
        unique_structs = np.unique(struct_keys)
        self.struct_map_ = {k: i for i, k in enumerate(unique_structs)}
        s_idx = np.array([self.struct_map_[k] for k in struct_keys], dtype=np.float64)
        
        # Koordinat 1D: memisahkan beda kelurahan sejauh 1e7 nomor urut
        self.coords_street_ = (w_idx * 1e7 + seq)[:, None]
        self.coords_struct_ = (s_idx * 1e7 + seq)[:, None]
        
        self.tree_street_ = cKDTree(self.coords_street_)
        self.tree_struct_ = cKDTree(self.coords_struct_)
        
        # Simpan one-hot target dan prior per old_ward_id
        self.y_onehot_ = np.eye(5)[y_num]
        self.global_prior_ = self.y_onehot_.mean(axis=0)
        
        df_tmp = pd.DataFrame({'w': old_ward, 's': struct_keys})
        for c in range(5):
            df_tmp[f'c_{c}'] = self.y_onehot_[:, c]
            
        w_grp = df_tmp.groupby('w')
        w_cnt = w_grp['c_0'].count()
        self.ward_prior_ = {}
        for c in range(5):
            self.ward_prior_[c] = ((w_grp[f'c_{c}'].sum() + 10.0 * self.global_prior_[c]) / (w_cnt + 10.0)).to_dict()
        return self

    def predict_proba(self, X_raw):
        b_id = X_raw['building_id'].astype(np.int64).values
        old_ward = b_id // 1_000_000
        seq = (b_id % 1_000_000).astype(np.float64)
        super_str = X_raw['superstructure'].astype(str).values
        
        w_idx = np.array([self.ward_map_.get(w, -999) for w in old_ward], dtype=np.float64)
        struct_keys = [f"{w}_{s}" for w, s in zip(old_ward, super_str)]
        s_idx = np.array([self.struct_map_.get(k, -999) for k in struct_keys], dtype=np.float64)
        
        q_street = (w_idx * 1e7 + seq)[:, None]
        q_struct = (s_idx * 1e7 + seq)[:, None]
        
        # 1. Tetangga terdekat di jalan yang sama
        d_st, idx_st = self.tree_street_.query(q_street, k=self.k_street, workers=-1)
        valid_st = (d_st < 1e6).astype(np.float64)
        w_st = valid_st / (1.0 + 0.25 * d_st)
        w_st_sum = w_st.sum(axis=1, keepdims=True)
        proba_st = np.einsum('ij,ijk->ik', w_st, self.y_onehot_[idx_st]) / np.maximum(w_st_sum, 1e-9)
        
        # 2. Tetangga terdekat di kelurahan yang sama & material superstructure yang sama
        d_su, idx_su = self.tree_struct_.query(q_struct, k=self.k_struct, workers=-1)
        valid_su = (d_su < 1e6).astype(np.float64)
        w_su = valid_su / (1.0 + 0.10 * d_su)
        w_su_sum = w_su.sum(axis=1, keepdims=True)
        proba_su = np.einsum('ij,ijk->ik', w_su, self.y_onehot_[idx_su]) / np.maximum(w_su_sum, 1e-9)
        
        # 3. Prior kelurahan mikro (old_ward_id)
        prior_w = np.column_stack([
            pd.Series(old_ward).map(self.ward_prior_[c]).fillna(self.global_prior_[c]).values
            for c in range(5)
        ])
        
        # Fallback ke prior jika tidak ada tetangga valid
        proba_st = np.where(w_st_sum > 0, proba_st, prior_w)
        proba_su = np.where(w_su_sum > 0, proba_su, prior_w)
        
        # Gabungan sinyal tetangga jalan (45%), tetangga sematerial (40%), dan prior kelurahan (15%)
        proba_total = 0.45 * proba_st + 0.40 * proba_su + 0.15 * prior_w
        return proba_total / proba_total.sum(axis=1, keepdims=True)

In [ ]:
# Model 1: LightGBM dengan max_bin=1023 agar ribuan old_ward_id & survey_seq terbaca tajam
lgbm_micro = lgb.LGBMClassifier(
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=85,
    max_depth=16,
    max_bin=1023,
    min_data_per_group=50,
    cat_smooth=20.0,
    colsample_bytree=0.75,
    subsample=0.8,
    subsample_freq=1,
    random_state=42,
    n_jobs=-1
)

# Model 2: XGBoost
xgb_micro = xgb.XGBClassifier(
    n_estimators=1100,
    learning_rate=0.03,
    max_depth=9,
    max_bin=512,
    subsample=0.8,
    colsample_bytree=0.75,
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

In [7]:
#VALIDASI LOKAL
micro_prep = MicroStreetPreprocessor()
X_extra_all = pd.concat([X_test_with_id, df_test_raw], axis=0, ignore_index=True)

print("Memproses Fitur Dekode building_id + Tetangga Jalan Sensus Penuh...")
micro_prep.fit(X_train_with_id, X_extra=X_extra_all)
X_tr_micro = micro_prep.transform(X_train_with_id)
X_te_micro = micro_prep.transform(X_test_with_id)
print(f"Total fitur sekarang: {X_tr_micro.shape[1]} fitur")

print("\nMelatih Model 1 (LightGBM max_bin=1023)...")
lgbm_micro.fit(X_tr_micro, y_train_num)
proba_lgb_m = lgbm_micro.predict_proba(X_te_micro)

print("Melatih Model 2 (XGBoost max_bin=512)...")
xgb_micro.fit(X_tr_micro, y_train_num)
proba_xgb_m = xgb_micro.predict_proba(X_te_micro)

print("Melatih Model 3 (StreetNeighborKNN cKDTree)...")
knn_micro = StreetNeighborKNN(k_street=20, k_struct=15)
knn_micro.fit(X_train_with_id, y_train_num)
proba_knn_m = knn_micro.predict_proba(X_test_with_id)

print(f"\n--- SKOR INDIVIDU (MURNI TANPA TUNING) ---")
print(f"LightGBM Murni        : {f1_score(y_test_num, np.argmax(proba_lgb_m, axis=1), average='macro'):.5f}")
print(f"XGBoost Murni         : {f1_score(y_test_num, np.argmax(proba_xgb_m, axis=1), average='macro'):.5f}")
print(f"StreetNeighborKNN     : {f1_score(y_test_num, np.argmax(proba_knn_m, axis=1), average='macro'):.5f}")

# Cari kombinasi 3 Model + Threshold terbaik
best_f1_m = 0
best_mix = None
best_w_m = None

for w_knn in [0.0, 0.08, 0.12, 0.16, 0.22]:
    rem = 1.0 - w_knn
    for ratio_lgb in [0.45, 0.50, 0.55]:
        w_lgb = rem * ratio_lgb
        w_xgb = rem * (1.0 - ratio_lgb)
        
        p_blend = w_lgb * proba_lgb_m + w_xgb * proba_xgb_m + w_knn * proba_knn_m
        res = minimize(
            lambda w: -f1_score(y_test_num, np.argmax(p_blend * w, axis=1), average='macro'),
            [0.59, 0.99, 0.95, 0.76, 0.58],
            method='Powell'
        )
        if -res.fun > best_f1_m:
            best_f1_m = -res.fun
            best_mix = (w_lgb, w_xgb, w_knn)
            best_w_m = res.x

print(f"\n==========================================")
print(f"Proporsi 3-Model Terbaik -> LGBM: {best_mix[0]:.3f} | XGB: {best_mix[1]:.3f} | StreetKNN: {best_mix[2]:.3f}")
print(f"Bobot Kelas Optimal      -> {np.round(best_w_m, 4)}")
print(f"Rekor Lokal Sebelumnya   : 0.56140")
print(f"Macro F1 Lokal SEKARANG  : {best_f1_m:.5f}")
print(f"==========================================")

final_preds_m = np.argmax((best_mix[0]*proba_lgb_m + best_mix[1]*proba_xgb_m + best_mix[2]*proba_knn_m) * best_w_m, axis=1)
print("\n--- CLASSIFICATION REPORT (MICRO-STREET + 3-MODEL ENSEMBLE) ---")
print(classification_report(y_test_formatted, ['Grade ' + str(p + 1) for p in final_preds_m]))

Contoh building_id berhasil dipulihkan: [366106001131, 304506000161, 313506000051]
Memproses Fitur Dekode building_id + Tetangga Jalan Sensus Penuh...
Total fitur sekarang: 55 fitur

Melatih Model 1 (LightGBM max_bin=1023)...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.109390 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41317
[LightGBM] [Info] Number of data points in the train set: 425013, number of used features: 55
[LightGBM] [Info] Start training from score -2.269613
[LightGBM] [Info] Start training from score -2.167201
[LightGBM] [Info] Start training from score -1.720821
[LightGBM] [Info] Start training from score -1.421545
[LightGBM] [Info] Start training fro

In [ ]:
# 1. PARAMETER OPTIMAL YANG SUDAH DIKUNCI DARI VALIDASI LOKAL (0.59402)
final_mix = (0.429, 0.351, 0.220)
final_class_weights = np.array([0.7933, 0.9358, 0.9130, 0.7600, 0.5338])

# Buang anomali age_building == 999 pada data latih
df_train_clean = df_train_raw[df_train_raw['age_building'] != 999].reset_index(drop=True)
del df_train_raw
gc.collect()

y_full_num = df_train_clean['damage_grade'].astype(str).str.replace('Grade ', '').astype(int).values - 1
X_full_with_id = df_train_clean.drop(columns=['damage_grade'], errors='ignore')
test_building_ids = df_test_raw['building_id'].copy()
del df_train_clean
gc.collect()

print(f"Total 100% data latih bersih: {len(X_full_with_id)} baris")

# 4. JALANKAN MODEL 3 (StreetNeighborKNN) LEBIH DULU & BEBASKAN RAM
print("Melatih & Memprediksi Model 3: StreetNeighborKNN (cKDTree) pada 100% data...")
knn_full = StreetNeighborKNN(k_street=20, k_struct=15)
knn_full.fit(X_full_with_id, y_full_num)
test_proba_knn = knn_full.predict_proba(df_test_raw)
joblib.dump(knn_full, "../models/model_knn_street.joblib", compress=3)
del knn_full
gc.collect()

# 5. PROSES 55 FITUR MIKRO-GEOSPASIAL PADA 100% DATA
print("Memproses 55 fitur MicroStreetPreprocessor pada 100% data...")
micro_prep_full = MicroStreetPreprocessor()
micro_prep_full.fit(X_full_with_id, X_extra=df_test_raw)

X_full_micro = micro_prep_full.transform(X_full_with_id)
X_test_kaggle_micro = micro_prep_full.transform(df_test_raw)

joblib.dump(micro_prep_full, "../models/preprocessor_microstreet.joblib", compress=3)
del X_full_with_id, df_test_raw, micro_prep_full
gc.collect()

# 6. LATIH LIGHTGBM & XGBOOST SECARA BERGANTIAN (HEMAT RAM)
print("Melatih Model 1: LightGBM (max_bin=1023) pada 100% data...")
lgbm_micro.fit(X_full_micro, y_full_num)
test_proba_lgb = lgbm_micro.predict_proba(X_test_kaggle_micro)
joblib.dump(lgbm_micro, "../models/model_lgbm_micro.joblib", compress=3)
del lgbm_micro
gc.collect()

print("Melatih Model 2: XGBoost (max_bin=512) pada 100% data...")
xgb_micro.fit(X_full_micro, y_full_num)
test_proba_xgb = xgb_micro.predict_proba(X_test_kaggle_micro)
joblib.dump(xgb_micro, "../models/model_xgb_micro.joblib", compress=3)
del xgb_micro, X_full_micro, X_test_kaggle_micro
gc.collect()

# 7. GABUNGKAN PROBABILITAS 3 MODEL
test_proba_blend = (
    final_mix[0] * test_proba_lgb +
    final_mix[1] * test_proba_xgb +
    final_mix[2] * test_proba_knn
)

final_kaggle_preds = np.argmax(test_proba_blend * final_class_weights, axis=1) + 1

submission_ultimate = pd.DataFrame({
    'building_id': test_building_ids,
    'damage_grade': final_kaggle_preds
})

print("\nPENGECEKAN WAJIB SEBELUM SUBMIT TERAKHIR")
print(f"Total baris       : {len(submission_ultimate)} (Wajib 228629)")
print(f"Jumlah nilai NaN  : {submission_ultimate.isna().sum().sum()} (Wajib 0)")
print("\nDISTRIBUSI PREDIKSI KAGGLE")
print(submission_ultimate['damage_grade'].value_counts())

submission_ultimate.to_csv("submission9_ultimate_microstreet_100pct.csv", index=False)
print("done!")

[1/5] Memuat train.csv dan test.csv...
Total 100% data latih bersih: 531267 baris

[2/5] Melatih & Memprediksi Model 3: StreetNeighborKNN (cKDTree) pada 100% data...

[3/5] Memproses 55 fitur MicroStreetPreprocessor pada 100% data...

[4/5] Melatih Model 1: LightGBM (max_bin=1023) pada 100% data...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.199145 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 41320
[LightGBM] [Info] Number of data points in the train set: 531267, number of used features: 55
[LightGBM] [Info] Start training from score -2.269606
[LightGBM] [Info] Start training from score -2.167198
[LightGBM] [Info] Start training from score -1.720820
[LightGBM] [Info

In [ ]:
ensemble_bundle = {
    'preprocessor': joblib.load("preprocessor_microstreet.joblib"),
    'lgbm_model': joblib.load("model_lgbm_micro.joblib"),
    'xgb_model': joblib.load("model_xgb_micro.joblib"),
    'knn_model': joblib.load("model_knn_street.joblib"),
    'model_weights': final_mix,
    'class_weights': final_class_weights
}
joblib.dump(ensemble_bundle, "../models/ensemble_lgbm_xgb_knn.joblib", compress=3)
del ensemble_bundle
gc.collect()